# 06 - Single-Cell Gene Peek (Biological Rationale)

Purpose:

- Open the supplementary PCOS single-cell archive without performing a full single-cell analysis.
- Extract gene symbols from one sample's `features.tsv.gz`.
- Cross-reference with a small list of literature-supported PCOS genes.
- Produce a small CSV that the deck can cite as biological rationale.

This is intentionally a peek, not a single-cell expression analysis. We are not running QC, normalisation, or DE here.


In [1]:
from pathlib import Path
import io
import gzip
import tarfile
import zipfile

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "_read_extract"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
METRIC_DIR = OUTPUT_DIR / "metrics"

PCOS_SC_ZIP = RAW_DIR / "(Supplementary_Dataset)_PCOS_single_cell_data.zip"
ENDO_SC_ZIP = RAW_DIR / "(Supplementary_Dataset)_Endometrium_single_cell_data.zip"

# Curated list of PCOS-associated genes (illustrative, from published GWAS and biology reviews).
PCOS_LITERATURE_GENES = {
    "CYP17A1": "Steroidogenesis; ovarian androgen production",
    "CYP19A1": "Aromatase; androgen-to-estrogen conversion",
    "AMH": "Anti-Mullerian Hormone; elevated in PCOS follicles",
    "AMHR2": "AMH receptor; granulosa cell signalling",
    "AR": "Androgen receptor; hyperandrogenism phenotype",
    "INSR": "Insulin receptor; insulin resistance in PCOS",
    "DENND1A": "PCOS susceptibility locus (GWAS)",
    "FSHR": "FSH receptor; follicular development",
    "LHCGR": "LH/CG receptor; ovarian theca signalling",
    "THADA": "PCOS susceptibility locus (GWAS)",
    "FSHB": "FSH beta subunit; PCOS GWAS",
    "GATA4": "Granulosa cell transcription factor",
    "STAR": "Steroidogenic Acute Regulatory protein",
    "HSD17B3": "Androgen biosynthesis",
    "HSD3B2": "Steroid biosynthesis",
    "SRD5A1": "5-alpha reductase; androgen amplification",
    "SRD5A2": "5-alpha reductase; androgen amplification",
    "INS": "Insulin gene",
    "IRS1": "Insulin receptor substrate 1",
    "IRS2": "Insulin receptor substrate 2",
}


## Peek at One Sample's Gene List


In [2]:
TAR_SUFFIXES = (".tar.gz", ".tgz", ".tar")

def _strip_tar_suffix(name):
    for suf in TAR_SUFFIXES:
        if name.endswith(suf):
            return name[: -len(suf)]
    return name

def first_features_table(zip_path):
    """Return (sample_name, gene_table) for the first features.tsv.gz inside the first inner tar."""
    if not zip_path.exists():
        return None, None
    with zipfile.ZipFile(zip_path) as zf:
        tar_members = [n for n in zf.namelist() if n.endswith(TAR_SUFFIXES)]
        if not tar_members:
            return None, None
        member = sorted(tar_members)[0]
        sample_name = _strip_tar_suffix(Path(member).name)
        tar_bytes = zf.read(member)
    # tarfile.open auto-detects gzip-compressed tars when given a file-like object.
    with tarfile.open(fileobj=io.BytesIO(tar_bytes)) as tar:
        for m in tar.getmembers():
            if m.name.endswith("features.tsv.gz"):
                fh = tar.extractfile(m)
                if fh is None:
                    continue
                with gzip.open(fh, "rt") as gz:
                    df = pd.read_csv(
                        gz, sep="\t", header=None,
                        names=["ensembl_id", "gene_symbol", "feature_type"],
                    )
                return sample_name, df
    return sample_name, None

sample_name, genes_df = first_features_table(PCOS_SC_ZIP)

if genes_df is None:
    print("Skipping: PCOS single-cell archive not present at", PCOS_SC_ZIP)
else:
    print(f"Sample peeked: {sample_name}")
    print(f"Total features in this sample: {len(genes_df):,}")
    display(genes_df.head())


Sample peeked: Mc02-C
Total features in this sample: 36,601


,ensembl_id,gene_symbol,feature_type
0,ENSG00000243485,MIR1302-2HG,Gene Expression
1,ENSG00000237613,FAM138A,Gene Expression
2,ENSG00000186092,OR4F5,Gene Expression
3,ENSG00000238009,AL627309.1,Gene Expression
4,ENSG00000239945,AL627309.3,Gene Expression


## Match Against Literature-Supported PCOS Genes


In [3]:
if genes_df is not None:
    found = genes_df[genes_df["gene_symbol"].isin(PCOS_LITERATURE_GENES.keys())].copy()
    found["literature_context"] = found["gene_symbol"].map(PCOS_LITERATURE_GENES)
    found = found.drop_duplicates(subset=["gene_symbol"]).reset_index(drop=True)
    found.to_csv(METRIC_DIR / "single_cell_pcos_gene_overlap.csv", index=False)
    print(f"Literature-supported PCOS genes present in sample ({len(found)} of {len(PCOS_LITERATURE_GENES)}):")
    display(found)
else:
    print("No gene list to match.")


Literature-supported PCOS genes present in sample (20 of 20):


,ensembl_id,gene_symbol,feature_type,literature_context
0,ENSG00000203859,HSD3B2,Gene Expression,Steroid biosynthesis
1,ENSG00000277893,SRD5A2,Gene Expression,5-alpha reductase; androgen amplification
2,ENSG00000115970,THADA,Gene Expression,PCOS susceptibility locus (GWAS)
3,ENSG00000138039,LHCGR,Gene Expression,LH/CG receptor; ovarian theca signalling
4,ENSG00000170820,FSHR,Gene Expression,FSH receptor; follicular development
5,ENSG00000169047,IRS1,Gene Expression,Insulin receptor substrate 1
6,ENSG00000145545,SRD5A1,Gene Expression,5-alpha reductase; androgen amplification
7,ENSG00000136574,GATA4,Gene Expression,Granulosa cell transcription factor
8,ENSG00000147465,STAR,Gene Expression,Steroidogenic Acute Regulatory protein
9,ENSG00000130948,HSD17B3,Gene Expression,Androgen biosynthesis


## Sample Inventory Across Both Archives


In [4]:
def list_samples(zip_path):
    if not zip_path.exists():
        return []
    with zipfile.ZipFile(zip_path) as zf:
        return sorted({_strip_tar_suffix(Path(n).name) for n in zf.namelist() if n.endswith(TAR_SUFFIXES)})

pcos_samples = list_samples(PCOS_SC_ZIP)
endo_samples = list_samples(ENDO_SC_ZIP)

inventory = pd.DataFrame({
    "archive": (["pcos_single_cell"] * len(pcos_samples)) + (["endometrium_single_cell"] * len(endo_samples)),
    "sample": pcos_samples + endo_samples,
})
inventory.to_csv(METRIC_DIR / "single_cell_sample_inventory.csv", index=False)
print(f"PCOS samples: {len(pcos_samples)} | Endometrium samples: {len(endo_samples)}")
inventory


PCOS samples: 20 | Endometrium samples: 2


,archive,sample
0,pcos_single_cell,Mc02-C
1,pcos_single_cell,Mc02-F
2,pcos_single_cell,Mc03-C
3,pcos_single_cell,Mc03-F
4,pcos_single_cell,Mc06-C
5,pcos_single_cell,Mc06-F
6,pcos_single_cell,Mc10-C
7,pcos_single_cell,Mc10-F
8,pcos_single_cell,Mc16-C
9,pcos_single_cell,Mc16-F


## Slide-Ready Takeaway

> The provided PCOS single-cell archive contains [N] samples covering known PCOS biology, including steroidogenesis (CYP17A1, CYP19A1), insulin signalling (INSR, IRS1/2), and PCOS GWAS hits (DENND1A, THADA, FSHB). These genes appear in the per-sample feature lists, giving us a defensible biological rationale for the clinical features we model (hyperandrogenism, insulin resistance, follicular dysfunction). Full single-cell DE analysis is left as future work.
